# 04 — Topic Modeling & Named Entity Recognition

## Objective
Identify **what** people are talking about (topics) and **who** they mention (players, teams, brands) in the YouTube comments.

## Approach
- **BERTopic**: Multilingual sentence embeddings → HDBSCAN clustering → c-TF-IDF topic representation. Topics are automatically labelled with their top-3 representative terms.
- **NER**: spaCy pipelines for Spanish and English. A custom dictionary of World Cup 2026 players, brands, and venues supplements the pre-trained models.

## Business value
A brand sponsor (e.g., Adidas, Coca-Cola) wants to know: *"Are people talking about us? Is the sentiment positive?"* Topic modeling reveals the *narrative* behind sentiment shifts.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.topic_modeling import (
    get_topic_info, add_topics_to_dataframe, add_entities_to_dataframe,
    extract_entities, KNOWN_PLAYERS
)
from src.utils import load_dataframe, save_dataframe
import pandas as pd
import matplotlib.pyplot as plt

### Load sentiment-enriched data

In [ ]:
df = load_dataframe(str(project_root / 'data' / 'processed' / 'sentiment.parquet'))
print(f"Loaded {len(df)} comments with sentiment")
display(df[['text_clean', 'sentiment_label', 'teams']].head(3))

### Named Entity Recognition

In [ ]:
sample_texts = df['text_clean'].dropna().head(5).tolist()
sample_langs = df.loc[:4, 'language'].tolist()

for txt, lang in zip(sample_texts, sample_langs):
    entities = extract_entities(txt, lang)
    spacy_ents = [e['text'] for e in entities['spacy']]
    custom_ents = [e['text'] for e in entities['custom']]
    print(f"[{lang}] {txt[:80]}...")
    print(f"  spaCy: {spacy_ents}")
    print(f"  custom: {custom_ents}")
    print()

### Run full NER pipeline

In [ ]:
ner_cached = project_root / 'data' / 'processed' / 'topic_ner.parquet'
if ner_cached.exists():
    df = load_dataframe(str(ner_cached))
else:
    df = add_entities_to_dataframe(df)
    save_dataframe(df, str(project_root / 'data' / 'processed' / 'topic_ner'), format='parquet')

if 'brands_mentioned' in df.columns:
    brand_mentions = df['brands_mentioned'][df['brands_mentioned'] != ''].str.split(',').explode().value_counts()
    print("Brand mentions:")
    display(brand_mentions)

    player_mentions = df['players_mentioned'][df['players_mentioned'] != ''].str.split(',').explode().value_counts()
    print("\nPlayer mentions:")
    display(player_mentions.head(20))

### Topic Modeling with BERTopic

In [ ]:
model_path = project_root / 'data' / 'processed' / 'bertopic_model'

if model_path.exists():
    from bertopic import BERTopic
    model = BERTopic.load(str(model_path))
    info = get_topic_info(model)
    print("Loaded existing BERTopic model")
else:
    df, model = add_topics_to_dataframe(
        df,
        model_save_path=model_path,
    )
    info = get_topic_info(model)

display(info[['Topic', 'topic_label', 'Count']].head(15))

### Topic distribution

In [ ]:
if 'topic_label' in df.columns:
    topic_counts = df['topic_label'].value_counts().head(12)
    topic_counts.plot(kind='barh', title='Top 12 Topics')
    plt.xlabel('Number of comments')
    plt.tight_layout()
    plt.show()
    
    cross = pd.crosstab(
        df['topic_label'],
        df['sentiment_label'],
        normalize='index',
    ).sort_values('positive', ascending=False)
    display(cross.head(10))

---
**Next step**: [05 — EDA & Insights](05_eda_insights.ipynb)